# 🫀 Project 09: Cardiovascular Disease Risk Engine with SHAP Interpretability
### Clinical Informatics, Imbalanced Classification & Explainable AI (SHAP)

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟡 Intermediate  
**Domain:** Healthcare & Cardiology  

---
### Notebook Outline:
1. **Environment Setup & Tooling**
2. **Clinical Data Ingestion & Imbalance Inspection**
3. **Exploratory Data Analysis: Lipid Ratios & Vascular Risk Factors**
4. **Machine Learning Pipeline: XGBoost with Class Balancing**
5. **ROC & Precision-Recall Diagnostic Evaluation**
6. **Model Interpretability: Global Feature Importances & SHAP Explanations**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Clinical diagnostics workspace ready.")

In [ ]:
# Ingestion & Preprocessing
df = pd.read_csv("data/cardiac_risk_clinical.csv")
print(f"Patients: {len(df)} | Event Prevalence: {df['cardiac_event_10yr'].mean():.2%}")

df['cholesterol_to_hdl_ratio'] = df['total_cholesterol'] / df['hdl_cholesterol']
df['gender_male'] = (df['gender'] == 'Male').astype(int)

features = ['age', 'gender_male', 'systolic_bp', 'total_cholesterol', 'hdl_cholesterol',
            'ldl_cholesterol', 'smoking_status', 'diabetes_status', 'bmi', 'cholesterol_to_hdl_ratio']
X = df[features]
y = df['cardiac_event_10yr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Training cases: {len(X_train)} | Test cases: {len(X_test)}")

In [ ]:
# XGBoost Training with Scale Pos Weight
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

clf = xgb.XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_weight, random_state=42, eval_metric='logloss'
)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:, 1]
print("=== Diagnostic Performance Metrics ===")
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, probs):.4f}")

In [ ]:
# Feature Attribution: XGBoost Gain & Weight Importance
imp = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
imp.plot(kind='barh', color='crimson')
plt.title("XGBoost Feature Importance (Gain Metric)", fontweight='bold')
plt.xlabel("Relative Contribution to Risk Scoring")
plt.show()